In [1]:
import pandas as pd

dataset = pd.read_csv("./data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

dataset.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print("Shape:", dataset.shape)

print("\nData Types:")
print(dataset.dtypes)

print("\nMissing Values:")
print(dataset.isnull().sum())

Shape: (7043, 21)

Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            

In [3]:
print("Unique blank/space values:")
print(dataset["TotalCharges"].value_counts().head())

print("\nNumber of whitespace-only values:")
print(dataset["TotalCharges"].str.strip().eq("").sum())

Unique blank/space values:
TotalCharges
         11
20.2     11
19.75     9
20.05     8
19.9      8
Name: count, dtype: int64

Number of whitespace-only values:
11


In [4]:
print("Duplicate rows:", dataset.duplicated().sum())
print("Duplicate customer IDs:", dataset["customerID"].duplicated().sum())

Duplicate rows: 0
Duplicate customer IDs: 0


In [5]:
blank_total_charges = dataset[
    dataset["TotalCharges"].str.strip() == ""
]

blank_total_charges[[
    "customerID",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "Churn"
]]

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


In [6]:
import numpy as np
# Replace whitespace-only values with NaN
dataset["TotalCharges"] = dataset["TotalCharges"].replace(r"^\s*$", np.nan, regex=True)

# Convert TotalCharges to numeric
dataset["TotalCharges"] = pd.to_numeric(dataset["TotalCharges"])

# Replace the 11 missing values with 0
dataset["TotalCharges"] = dataset["TotalCharges"].fillna(0)

# Verify
print(dataset["TotalCharges"].dtype)
print("Missing values:", dataset["TotalCharges"].isna().sum())

float64
Missing values: 0


In [7]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
churn_counts = dataset["Churn"].value_counts()

print(churn_counts)

print("\nChurn Percentage:")
print(dataset["Churn"].value_counts(normalize=True).mul(100).round(2))

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn Percentage:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


In [9]:
total_customers = dataset["customerID"].nunique()
churned_customers = (dataset["Churn"] == "Yes").sum()
retained_customers = (dataset["Churn"] == "No").sum()

churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

print(f"Total Customers: {total_customers:,}")
print(f"Churned Customers: {churned_customers:,}")
print(f"Retained Customers: {retained_customers:,}")
print(f"Churn Rate: {churn_rate:.2f}%")
print(f"Retention Rate: {retention_rate:.2f}%")

Total Customers: 7,043
Churned Customers: 1,869
Retained Customers: 5,174
Churn Rate: 26.54%
Retention Rate: 73.46%


In [10]:
#How long do customers typically stay active?

In [11]:
print(dataset["tenure"].describe())

count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64


In [12]:
average_tenure = dataset["tenure"].mean()
median_tenure = dataset["tenure"].median()

print(f"Average Tenure: {average_tenure:.2f} months")
print(f"Median Tenure: {median_tenure:.2f} months")

Average Tenure: 32.37 months
Median Tenure: 29.00 months


In [13]:
tenure_by_churn = dataset.groupby("Churn")["tenure"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

tenure_by_churn

,count,mean,median,min,max
Churn,,,,,
No,5174,37.57,38.0,0,72
Yes,1869,17.98,10.0,1,72


Initial insight

Churned customers have an average tenure of only 18 months, compared with approximately 38 months for retained customers.

Even more interestingly, the median churned customer stayed only 10 months, while the median retained customer stayed 38 months.

This strongly suggests that early customer lifecycle is a major retention challenge.

In [14]:
tenure_bins = [0, 6, 12, 24, 48, 72]
tenure_labels = [
    "0-6 months",
    "7-12 months",
    "13-24 months",
    "25-48 months",
    "49-72 months"
]

dataset["TenureGroup"] = pd.cut(
    dataset["tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    include_lowest=True
)

In [15]:
tenure_distribution = dataset["TenureGroup"].value_counts().sort_index()

tenure_distribution

TenureGroup
0-6 months      1481
7-12 months      705
13-24 months    1024
25-48 months    1594
49-72 months    2239
Name: count, dtype: int64

In [16]:
tenure_churn = pd.crosstab(
    dataset["TenureGroup"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

tenure_churn

Churn,No,Yes
TenureGroup,,
0-6 months,47.06,52.94
7-12 months,64.11,35.89
13-24 months,71.29,28.71
25-48 months,79.61,20.39
49-72 months,90.49,9.51


In [17]:
# Analyze churn by contract type

contract_churn = pd.crosstab(
    dataset["Contract"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

contract_churn

Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


In [18]:
contract_counts = pd.crosstab(
    dataset["Contract"],
    dataset["Churn"]
)

contract_counts

Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [19]:
month_to_month_churned = (
    (dataset["Contract"] == "Month-to-month") &
    (dataset["Churn"] == "Yes")
).sum()

share_of_total_churn = month_to_month_churned / churned_customers * 100

print(f"Month-to-month churned customers: {month_to_month_churned:,}")
print(f"Share of all churned customers: {share_of_total_churn:.2f}%")

Month-to-month churned customers: 1,655
Share of all churned customers: 88.55%


In [20]:
tenure_contract_churn = pd.crosstab(
    dataset["TenureGroup"],
    dataset["Contract"],
    normalize="index"
).mul(100).round(2)

tenure_contract_churn

Contract,Month-to-month,One year,Two year
TenureGroup,,,
0-6 months,95.41,2.63,1.96
7-12 months,82.41,12.06,5.53
13-24 months,71.97,19.24,8.79
25-48 months,50.31,32.50,17.19
49-72 months,15.27,28.32,56.41


In [21]:
early_customers = dataset[dataset["TenureGroup"] == "0-6 months"]

early_contract_churn = pd.crosstab(
    early_customers["Contract"],
    early_customers["Churn"],
    normalize="index"
).mul(100).round(2)

early_contract_churn

Churn,No,Yes
Contract,,
Month-to-month,44.80,55.20
One year,89.74,10.26
Two year,100.00,0.00


In [22]:
payment_churn = pd.crosstab(
    dataset["PaymentMethod"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

payment_churn

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.29,16.71
Credit card (automatic),84.76,15.24
Electronic check,54.71,45.29
Mailed check,80.89,19.11


In [23]:
payment_counts = pd.crosstab(
    dataset["PaymentMethod"],
    dataset["Churn"]
)

payment_counts

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),1286,258
Credit card (automatic),1290,232
Electronic check,1294,1071
Mailed check,1304,308


In [24]:
electronic_check_churned = (
    (dataset["PaymentMethod"] == "Electronic check") &
    (dataset["Churn"] == "Yes")
).sum()

electronic_check_share = electronic_check_churned / churned_customers * 100

print(f"Electronic check churned customers: {electronic_check_churned:,}")
print(f"Share of all churned customers: {electronic_check_share:.2f}%")

Electronic check churned customers: 1,071
Share of all churned customers: 57.30%


In [25]:
internet_churn = pd.crosstab(
    dataset["InternetService"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

internet_churn

Churn,No,Yes
InternetService,,
DSL,81.04,18.96
Fiber optic,58.11,41.89
No,92.60,7.40


In [26]:
internet_counts = pd.crosstab(
    dataset["InternetService"],
    dataset["Churn"]
)

internet_counts

Churn,No,Yes
InternetService,,
DSL,1962,459
Fiber optic,1799,1297
No,1413,113


In [27]:
fiber_churned = (
    (dataset["InternetService"] == "Fiber optic") &
    (dataset["Churn"] == "Yes")
).sum()

fiber_churn_share = fiber_churned / churned_customers * 100

print(f"Fiber optic churned customers: {fiber_churned:,}")
print(f"Share of all churned customers: {fiber_churn_share:.2f}%")

Fiber optic churned customers: 1,297
Share of all churned customers: 69.40%


In [28]:
monthly_charges_churn = dataset.groupby("Churn")["MonthlyCharges"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

monthly_charges_churn

,count,mean,median,min,max
Churn,,,,,
No,5174,61.27,64.43,18.25,118.75
Yes,1869,74.44,79.65,18.85,118.35


In [29]:
charge_bins = [0, 30, 60, 90, 120, float("inf")]
charge_labels = [
    "<$30",
    "$30-$60",
    "$60-$90",
    "$90-$120",
    ">$120"
]

dataset["MonthlyChargeGroup"] = pd.cut(
    dataset["MonthlyCharges"],
    bins=charge_bins,
    labels=charge_labels,
    include_lowest=True
)

In [30]:
charge_churn = pd.crosstab(
    dataset["MonthlyChargeGroup"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

charge_churn

Churn,No,Yes
MonthlyChargeGroup,,
<$30,90.20,9.80
$30-$60,74.07,25.93
$60-$90,66.09,33.91
$90-$120,67.22,32.78


In [31]:
charge_counts = pd.crosstab(
    dataset["MonthlyChargeGroup"],
    dataset["Churn"]
)

charge_counts

Churn,No,Yes
MonthlyChargeGroup,,
<$30,1491,162
$30-$60,937,328
$60-$90,1577,809
$90-$120,1169,570


In [32]:
tenure_contract = pd.crosstab(
    dataset["TenureGroup"],
    dataset["Contract"],
    values=dataset["Churn"].eq("Yes"),
    aggfunc="mean"
).mul(100).round(2)

tenure_contract

Contract,Month-to-month,One year,Two year
TenureGroup,,,
0-6 months,55.20,10.26,0.00
7-12 months,42.00,10.59,0.00
13-24 months,37.72,8.12,0.00
25-48 months,32.92,10.62,2.19
49-72 months,26.02,12.93,3.33


In [33]:
senior_churn = pd.crosstab(
    dataset["SeniorCitizen"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

senior_churn

Churn,No,Yes
SeniorCitizen,,
0,76.39,23.61
1,58.32,41.68


In [34]:
senior_churn.index = ["Non-Senior", "Senior"]

senior_churn

Churn,No,Yes
Non-Senior,76.39,23.61
Senior,58.32,41.68


In [35]:
senior_counts = pd.crosstab(
    dataset["SeniorCitizen"],
    dataset["Churn"]
)

senior_counts.index = ["Non-Senior", "Senior"]

senior_counts

Churn,No,Yes
Non-Senior,4508,1393
Senior,666,476


In [36]:
partner_churn = pd.crosstab(
    dataset["Partner"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

partner_churn

Churn,No,Yes
Partner,,
No,67.04,32.96
Yes,80.34,19.66


In [37]:
dependents_churn = pd.crosstab(
    dataset["Dependents"],
    dataset["Churn"],
    normalize="index"
).mul(100).round(2)

dependents_churn

Churn,No,Yes
Dependents,,
No,68.72,31.28
Yes,84.55,15.45


In [38]:
partner_counts = pd.crosstab(
    dataset["Partner"],
    dataset["Churn"]
)

dependents_counts = pd.crosstab(
    dataset["Dependents"],
    dataset["Churn"]
)

print("Partner:")
print(partner_counts)

print("\nDependents:")
print(dependents_counts)

Partner:
Churn      No   Yes
Partner            
No       2441  1200
Yes      2733   669

Dependents:
Churn         No   Yes
Dependents            
No          3390  1543
Yes         1784   326


In [40]:
risk_profile = (
    dataset.groupby(
        ["TenureGroup", "Contract", "PaymentMethod"],
        observed=True
    )
    .agg(
        Customers=("customerID", "nunique"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

risk_profile["ChurnRate"] = (
    risk_profile["Churned"] /
    risk_profile["Customers"] * 100
).round(2)

risk_profile.sort_values(
    ["ChurnRate", "Customers"],
    ascending=[False, False]
).head(15)

,TenureGroup,Contract,PaymentMethod,Customers,Churned,ChurnRate
2,0-6 months,Month-to-month,Electronic check,680,460,67.65
0,0-6 months,Month-to-month,Bank transfer (automatic),110,63,57.27
13,7-12 months,Month-to-month,Electronic check,274,142,51.82
25,13-24 months,Month-to-month,Electronic check,355,182,51.27
1,0-6 months,Month-to-month,Credit card (automatic),119,53,44.54
37,25-48 months,Month-to-month,Electronic check,376,154,40.96
3,0-6 months,Month-to-month,Mailed check,504,204,40.48
11,7-12 months,Month-to-month,Bank transfer (automatic),98,38,38.78
12,7-12 months,Month-to-month,Credit card (automatic),72,26,36.11
49,49-72 months,Month-to-month,Electronic check,165,56,33.94


In [41]:
risk_profile_4d = (
    dataset.groupby(
        ["TenureGroup", "Contract", "PaymentMethod", "InternetService"],
        observed=True
    )
    .agg(
        Customers=("customerID", "nunique"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

risk_profile_4d["ChurnRate"] = (
    risk_profile_4d["Churned"] /
    risk_profile_4d["Customers"] * 100
).round(2)

risk_profile_4d.sort_values(
    ["ChurnRate", "Customers"],
    ascending=[False, False]
).head(20)

,TenureGroup,Contract,PaymentMethod,InternetService,Customers,Churned,ChurnRate
16,0-6 months,One year,Electronic check,Fiber optic,1,1,100.00
131,49-72 months,Month-to-month,Mailed check,No,1,1,100.00
1,0-6 months,Month-to-month,Bank transfer (automatic),Fiber optic,41,38,92.68
7,0-6 months,Month-to-month,Electronic check,Fiber optic,440,332,75.45
26,7-12 months,Month-to-month,Bank transfer (automatic),Fiber optic,46,33,71.74
4,0-6 months,Month-to-month,Credit card (automatic),Fiber optic,42,29,69.05
35,7-12 months,Month-to-month,Mailed check,Fiber optic,27,17,62.96
10,0-6 months,Month-to-month,Mailed check,Fiber optic,96,60,62.50
32,7-12 months,Month-to-month,Electronic check,Fiber optic,191,117,61.26
61,13-24 months,Month-to-month,Electronic check,Fiber optic,255,150,58.82


In [42]:
dataset.to_csv(
    "telco_churn_analysis_ready.csv",
    index=False
)

print("File saved successfully.")
print(dataset.shape)

File saved successfully.
(7043, 23)


In [43]:
print(dataset.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'TenureGroup', 'MonthlyChargeGroup']
